# 03 — Missing-value imputation

Runs `src/preprocessing/missing_values.impute_missing_values`, ported from
`missing-value-imputation.ipynb`. Input is the outlier-treated table
(`gurgaon_properties_outlier_treated.csv`, 24 cols); output is
`gurgaon_properties_missing_value_imputation.csv` (18 cols, no missing values).

Unlike the outlier and feature-selection stages, this notebook **does** fully
reproduce its output: the port matches the committed file **cell-for-cell
(63 972 / 63 972), no deviations**. A demonstration of already-tested code — no
new logic.

In [1]:
import sys, logging
from pathlib import Path

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

# impute_missing_values logs each step (row counts, cascade progress).
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

from src.preprocessing.missing_values import impute_missing_values

INTERIM = REPO_ROOT / "data" / "interim"
PROCESSED = REPO_ROOT / "data" / "processed"


## What `impute_missing_values` does

In notebook order:

1. **`built_up_area`** (null in ~55 % of rows) — back-derive from
   `super_built_up_area` / `carpet_area` using the ratios `1.105` / `0.9`
   (the medians of the rows where all three areas are present). Then an
   anomaly override: rows with `built_up_area < 2000` **and** `price > 2.5` Cr
   have `built_up_area` replaced by the raw `area`.
2. **`floorNum`** — the 17 null rows get a flat `2.0` (see the weak-spot note).
3. **null `society`** — the one such row is dropped (not imputed).
4. **`agePossession`** — the `"Undefined"` sentinel (291 rows) is filled by a
   cascade of group modes: within `(sector, property_type)`, then `(sector,)`,
   then `(property_type,)`.

Six columns are then dropped (`area`, `areaWithType`, `super_built_up_area`,
`carpet_area`, `area_room_ratio`, `facing`), 24 → 18.

In [2]:
treated = pd.read_csv(INTERIM / "gurgaon_properties_outlier_treated.csv")
print("input:", treated.shape)
print("\nnulls in the input (columns with any):")
print(treated.isnull().sum()[treated.isnull().sum() > 0])

input: (3555, 24)

nulls in the input (columns with any):
society                   1
floorNum                 17
facing                 1011
super_built_up_area    1680
built_up_area          1968
carpet_area            1715
dtype: int64


In [3]:
imputed = impute_missing_values(treated)
print("\ninput  :", treated.shape, " nulls:", int(treated.isnull().sum().sum()))
print("output :", imputed.shape, " nulls:", int(imputed.isnull().sum().sum()))

# `imputed` is row-aligned with the input minus the one dropped null-society
# row; keep a matching copy of the input for the row-level comparisons below.
treated_kept = treated[treated["society"].notna()].reset_index(drop=True)

built_up_area: filled 1968 null(s), anomaly-overrode 275 row(s)


floorNum: filled 17 null(s) with 2.0


dropping 1 row(s) with null society (positions [2536])


agePossession: after (sector, property_type) mode-fill, 55 still 'Undefined'


agePossession: after (sector) mode-fill, 29 still 'Undefined'


agePossession: after (property_type) mode-fill, 0 still 'Undefined'



input  : (3555, 24)  nulls: 6392
output : (3554, 18)  nulls: 0


In [4]:
imputed.head()

,property_type,society,sector,price,price_per_sqft,bedRoom,bathroom,balcony,floorNum,agePossession,built_up_area,study room,servant room,store room,pooja room,others,furnishing_type,luxury_score
0,flat,signature global park 4,sector 36,0.82,7586.0,3.0,2.0,2,2.0,New Property,850.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
1,flat,smart world gems,sector 89,0.95,8597.0,2.0,2.0,2,4.0,New Property,1226.0,1.0,1.0,0.0,0.0,0.0,0.0,38.0
2,flat,breez global hill view,sohna road,0.32,5470.0,2.0,2.0,1,17.0,New Property,1000.0,0.0,0.0,0.0,0.0,0.0,0.0,49.0
3,flat,bestech park view sanskruti,sector 92,1.60,8020.0,3.0,4.0,3+,10.0,Relatively New,1615.0,0.0,1.0,0.0,0.0,1.0,1.0,174.0
4,flat,suncity avenue,sector 102,0.48,9023.0,2.0,2.0,1,5.0,Relatively New,582.0,0.0,0.0,1.0,0.0,0.0,0.0,159.0


In [5]:
# exact-match check against the committed processed file
expected = pd.read_csv(PROCESSED / "gurgaon_properties_missing_value_imputation.csv")
assert list(imputed.columns) == list(expected.columns)
assert imputed.shape == expected.shape
total = match = 0
for col in expected.columns:
    a, b = imputed[col], expected[col]
    m = (np.isclose(a.astype(float), b.astype(float), equal_nan=True)
         if b.dtype.kind in "fi" else a.astype(str) == b.astype(str))
    total += len(m); match += int(m.sum())
print(f"cell match vs committed file: {match}/{total} = {100 * match / total:.4f}%")

cell match vs committed file: 63972/63972 = 100.0000%


## The `agePossession` cascade, row by row

The step logs above show `291 → 55 → 29 → 0` — 291 `"Undefined"` values,
whittled down by the three group-mode passes. A few of them, with what each
was filled to:

In [6]:
undef = treated_kept["agePossession"].eq("Undefined").values
sample = pd.DataFrame({
    "sector": treated_kept.loc[undef, "sector"].values,
    "property_type": treated_kept.loc[undef, "property_type"].values,
    "was": "Undefined",
    "filled_to": imputed.loc[undef, "agePossession"].values,
})
print(f"{undef.sum()} 'Undefined' rows, all now filled:",
      sorted(sample["filled_to"].unique()))
sample.head(12)

291 'Undefined' rows, all now filled: ['Moderately Old', 'New Property', 'Old Property', 'Relatively New', 'Under Construction']


,sector,property_type,was,filled_to
0,sector 109,house,Undefined,Relatively New
1,sector 89,house,Undefined,New Property
2,sector 89,flat,Undefined,New Property
3,sector 102,flat,Undefined,Relatively New
4,sector 3,house,Undefined,Moderately Old
5,sector 105,house,Undefined,Moderately Old
6,sector 55,house,Undefined,Old Property
7,sector 84,flat,Undefined,Relatively New
8,sector 4,house,Undefined,Moderately Old
9,sector 78,flat,Undefined,Moderately Old


## `built_up_area` — fill + anomaly override

1 968 nulls in, 0 out. The step log above reports **275 anomaly-override rows**
(`built_up_area < 2000` and `price > 2.5` Cr → `built_up_area = area`). On many
of those `built_up_area` already equalled `area`, so the override is a no-op;
below are the rows where it actually changed a value.

In [7]:
print("built_up_area nulls:",
      int(treated["built_up_area"].isnull().sum()), "->",
      int(imputed["built_up_area"].isnull().sum()))

changed = treated_kept["built_up_area"].values != imputed["built_up_area"].values
# ignore rows that only changed because they were NaN and got ratio-imputed
overridden = changed & treated_kept["built_up_area"].notna().values & (
    imputed["built_up_area"].values == treated_kept["area"].values
)
print("rows where the override changed built_up_area to equal raw area:",
      int(overridden.sum()))

pd.DataFrame({
    "price": imputed.loc[overridden, "price"],
    "built_up_area (was)": treated_kept.loc[overridden, "built_up_area"].values,
    "built_up_area (now)": imputed.loc[overridden, "built_up_area"].values,
    "raw area": treated_kept.loc[overridden, "area"].values,
}).head(10)

built_up_area nulls: 1968 -> 0
rows where the override changed built_up_area to equal raw area: 111


,price,built_up_area (was),built_up_area (now),raw area
35,8.25,300.0,2160.0,2160.0
58,5.75,260.0,2430.0,2430.0
68,7.60,1935.0,1961.0,1961.0
73,2.65,1800.0,1400.0,1400.0
95,3.65,1869.0,2395.0,2395.0
145,11.00,500.0,4125.0,4125.0
205,2.75,1750.0,1975.0,1975.0
280,3.80,162.0,1458.0,1458.0
283,11.30,480.0,4500.0,4500.0
322,4.50,360.0,3240.0,3240.0


## Weak spots (documented in the module, not silently applied)

- **`floorNum` flat fill.** All 17 null `floorNum` rows get `2.0`, ignoring
  `property_type` — a house floor is typically 0–1. Kept as the default;
  `impute_missing_values(df, floornum_fill=...)` overrides it.
- **The anomaly override is blunt** — it fixes genuine garbage (300 sqft at
  ₹8 Cr) but also shifts some borderline rows where `built_up_area` looked
  plausible.
- **`price_per_sqft` is left stale** after `built_up_area` changes — it was
  derived upstream and isn't recomputed here (the committed file has it stale
  too; re-deriving it isn't this stage's job).
- **The null-`society` row is dropped, not imputed** — ported by content
  (`society` is null) rather than the notebook's hardcoded `df.drop(index=[2536])`.